In [3]:
"""Predict continuous MIC values using XGBoost regression."""

import os
from itertools import product

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
)
from sklearn.metrics import r2_score


# Define project and input/output directories.
# Run the script from the repository root.
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
input_dir = os.path.join(base_dir, "data")
results_dir = os.path.join(base_dir, "results")


# Configure the number of independent runs and the test set proportion.
n_runs = 100
test_size = 0.20


# Identify all CSV datasets in the input directory.
dataset_files = [
    f for f in os.listdir(input_dir)
    if f.lower().endswith(".csv")
]
dataset_files.sort()

print(f"Found {len(dataset_files)} datasets:")
for f in dataset_files:
    print(" -", f)


# Process each dataset independently.
for dataset_file in dataset_files:

    print("\n" + "=" * 80)
    print(f"Processing dataset: {dataset_file}")
    print("=" * 80)

    dataset_name = os.path.splitext(dataset_file)[0]
    dataset_path = os.path.join(input_dir, dataset_file)
    df = pd.read_csv(dataset_path)

    # Check that the required columns are present.
    required_columns = ["sample_id", "outcome"]
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(
            f"WARNING: {dataset_file} is missing required "
            f"columns: {missing_columns}"
        )
        print("Dataset skipped.")
        continue

    # Remove rows with missing sample identifiers or MIC values.
    df = df.dropna(subset=["sample_id", "outcome"]).copy()

    # Convert MIC values to numeric format and remove invalid entries.
    df["outcome"] = pd.to_numeric(df["outcome"], errors="coerce")
    df = df.dropna(subset=["outcome"]).copy()

    # Retain only MIC values observed in at least two samples.
    mic_counts = df["outcome"].value_counts()
    valid_mics = mic_counts[mic_counts > 1].index
    df = df[df["outcome"].isin(valid_mics)].copy()

    # Use all columns except the sample identifier and MIC target as features.
    feature_columns = [
        col for col in df.columns
        if col not in ["sample_id", "outcome"]
    ]

    # Check that the dataset contains enough observations for the analysis.
    if df.empty:
        print(
            f"WARNING: {dataset_file} contains no valid observations "
            "after filtering. Dataset skipped."
        )
        continue

    if not feature_columns:
        print(
            f"WARNING: {dataset_file} contains no predictor features. "
            "Dataset skipped."
        )
        continue

    if df["outcome"].nunique() < 2:
        print(
            f"WARNING: {dataset_file} contains fewer than two distinct "
            "MIC values after filtering. Dataset skipped."
        )
        continue

    # Initialize containers for all outputs.
    global_accuracy_results = []
    predictions_results = []
    per_dilution_results = []
    importance_results = []

    # Repeat the evaluation using a different random seed for each run.
    for run in range(1, n_runs + 1):

        print(f"{dataset_name} - run {run}/{n_runs}")

        random_seed = run

        # Split the dataset into training and test sets.
        # Stratification preserves the distribution of observed MIC values.
        train_df, test_df = train_test_split(
            df,
            test_size=test_size,
            stratify=df["outcome"],
            random_state=random_seed,
        )

        # Separate predictor features, MIC targets, and sample identifiers.
        X_train = train_df[feature_columns].copy()
        y_train = train_df["outcome"].copy()

        X_test = test_df[feature_columns].copy()
        y_test = test_df["outcome"].copy()

        sample_ids_test = test_df["sample_id"].copy()

        # Define the factorial design for initial hyperparameter evaluation.
        factorial_params = {
            "max_depth": [1, 6],
            "colsample_bytree": [0.25, 1],
            "subsample": [0.25, 1],
        }

        combinations = list(product(*factorial_params.values()))
        factorial_results = []

        # Evaluate each combination using five-fold cross-validation and R².
        for combo in combinations:

            params = dict(zip(factorial_params.keys(), combo))

            model = XGBRegressor(
                objective="reg:squarederror",
                n_estimators=100,
                learning_rate=0.1,
                random_state=random_seed,
                n_jobs=-1,
                verbosity=0,
                **params,
            )

            r2_scores = cross_val_score(
                model,
                X_train,
                y_train,
                cv=5,
                scoring="r2",
                n_jobs=1,
            )

            mean_r2 = np.mean(r2_scores)
            factorial_results.append((params, mean_r2))

        # Select the factorial combination with the highest mean R².
        best_factorial = max(
            factorial_results,
            key=lambda x: x[1],
        )[0]

        # Define the grid search using the selected subsampling parameters.
        param_grid = {
            "max_depth": [3, 4, 5, 6],
            "learning_rate": [0.01, 0.05, 0.1],
            "colsample_bytree": [
                best_factorial["colsample_bytree"]
            ],
            "subsample": [
                best_factorial["subsample"]
            ],
            "n_estimators": [100, 600],
        }

        # Initialize the XGBoost regressor for hyperparameter optimization.
        grid_model = XGBRegressor(
            objective="reg:squarederror",
            random_state=random_seed,
            n_jobs=-1,
            verbosity=0,
        )

        # Select the final hyperparameters using five-fold cross-validation.
        grid_search = GridSearchCV(
            estimator=grid_model,
            param_grid=param_grid,
            scoring="r2",
            cv=5,
            n_jobs=1,
            verbose=0,
        )

        grid_search.fit(X_train, y_train)

        # Retrieve the best-performing model from the grid search.
        best_model = grid_search.best_estimator_

        # Calculate feature importance.
        # The existing long-format output is retained unchanged.
        feature_importances = pd.Series(
            best_model.feature_importances_,
            index=X_train.columns,
        ).sort_values(ascending=False)

        importance_run_df = pd.DataFrame({
            "run": run,
            "feature": feature_importances.index,
            "importance": feature_importances.values,
        })

        importance_results.append(importance_run_df)

        # Predict continuous MIC values for the test set.
        y_pred = best_model.predict(X_test)

        # A prediction agrees if its absolute error is no greater than
        # one log2 MIC dilution.
        agreement = (
            np.abs(y_test.to_numpy() - y_pred) <= 1
        ).astype(int)

        # Calculate global accuracy and R².
        accuracy_1_diluition = agreement.mean()
        r2 = r2_score(y_test, y_pred)

        # Store only the requested global metrics.
        global_accuracy_results.append({
            "run": run,
            "accuracy_1_diluition": accuracy_1_diluition,
            "R2": r2,
        })

        # Store predictions and agreement for each test sample.
        pred_run_df = pd.DataFrame({
            "run": run,
            "sample_id": sample_ids_test.values,
            "real_mic": y_test.values,
            "predicted_mic": y_pred,
            "agreement": agreement,
        })

        predictions_results.append(pred_run_df)

        # Calculate accuracy separately for each real MIC value.
        # predicted_mic is the mean prediction for that MIC group.
        per_dil_run = (
            pred_run_df
            .groupby(["run", "real_mic"], as_index=False)
            .agg(
                predicted_mic=("predicted_mic", "mean"),
                n_samples=("real_mic", "size"),
                accuracy_1_diluition=("agreement", "mean"),
            )
        )

        # Arrange columns in the requested order.
        per_dil_run = per_dil_run[
            [
                "run",
                "real_mic",
                "predicted_mic",
                "n_samples",
                "accuracy_1_diluition",
            ]
        ]

        per_dilution_results.append(per_dil_run)

    # Combine the results from all runs.
    global_accuracy_df = pd.DataFrame(global_accuracy_results)

    predictions_df = pd.concat(
        predictions_results,
        axis=0,
        ignore_index=True,
    )

    per_dilution_df = pd.concat(
        per_dilution_results,
        axis=0,
        ignore_index=True,
    )

    importance_df = pd.concat(
        importance_results,
        axis=0,
        ignore_index=True,
    )

    # Arrange columns in the requested order.
    global_accuracy_df = global_accuracy_df[
        ["run", "accuracy_1_diluition", "R2"]
    ]

    predictions_df = predictions_df[
        [
            "run",
            "sample_id",
            "real_mic",
            "predicted_mic",
            "agreement",
        ]
    ]

    # Define output filenames.
    global_accuracy_output_path = os.path.join(
        results_dir,
        f"accuracy_global_{dataset_name}.csv",
    )

    predictions_output_path = os.path.join(
        results_dir,
        f"prediction_per_sample_{dataset_name}.csv",
    )

    per_dilution_output_path = os.path.join(
        results_dir,
        f"accuracy_per_dilution_{dataset_name}.csv",
    )

    importance_output_path = os.path.join(
        results_dir,
        f"feature_importance_{dataset_name}.csv",
    )

    # Save the output tables directly in results/.
    global_accuracy_df.to_csv(
        global_accuracy_output_path,
        index=False,
    )

    predictions_df.to_csv(
        predictions_output_path,
        index=False,
    )

    per_dilution_df.to_csv(
        per_dilution_output_path,
        index=False,
    )

    importance_df.to_csv(
        importance_output_path,
        index=False,
    )

    print(f"\nFiles saved for {dataset_name}:")
    print(" - Global accuracy:       ", global_accuracy_output_path)
    print(" - Predictions per sample:", predictions_output_path)
    print(" - Accuracy per MIC:      ", per_dilution_output_path)
    print(" - Feature importance:    ", importance_output_path)

print("\nProcessing completed.")

Found 1 datasets:
 - xgboost_test_dataset.csv

Processing dataset: xgboost_test_dataset.csv
xgboost_test_dataset - run 1/2
xgboost_test_dataset - run 2/2

Files saved for xgboost_test_dataset:
 - Global accuracy:        /dati/Massi/script_github/results/accuracy_global_xgboost_test_dataset.csv
 - Predictions per sample: /dati/Massi/script_github/results/prediction_per_sample_xgboost_test_dataset.csv
 - Accuracy per MIC:       /dati/Massi/script_github/results/accuracy_per_dilution_xgboost_test_dataset.csv
 - Feature importance:     /dati/Massi/script_github/results/feature_importance_xgboost_test_dataset.csv

Processing completed.
